# 실험 02: Pydantic과 실무 제약조건 (PydanticOutputParser)

## 1. 실험 제목과 목표
- **제목**: Pydantic을 활용한 엄격한 타입 검증 및 제약조건 부여
- **목표**: 파이썬의 표준 데이터 검증 라이브러리인 Pydantic을 이용해 LLM의 응답 스키마(Schema)를 강제합니다. 특히 `Enum`과 `Field`를 사용해 특정 단어만 뱉게 만드는 실무 기법을 배웁니다.

## 2. 실험 개요
1. **실무 실험 1**: 리뷰 분석기 (별점 1~5점 제한, 감정 상태 Enum 제한)
2. **실패/주의 케이스**: 모델이 Enum 목록에 없는 이상한 값을 내뱉거나, 숫자가 들어가야 할 곳에 문자를 넣었을 때 터지는 `ValidationError` 관찰

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv
from enum import Enum
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 실무 실험 1: Pydantic 설계도(Schema) 작성
LLM에게 자유를 주지 않고, 우리가 정의한 규칙 안에서만 대답하도록 가둡니다.

In [3]:
# 1. Enum을 통한 범주형 데이터 강제
class Sentiment(str, Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"

# 2. Pydantic BaseModel 작성
class ReviewAnalysis(BaseModel):
    # Field(description=...) 은 LLM에게 이 항목을 어떻게 채울지 알려주는 아주 중요한 프롬프트입니다.
    summary: str = Field(description="리뷰의 핵심 내용을 1문장으로 요약")
    sentiment: Sentiment = Field(description="리뷰의 감정 상태. 반드시 지정된 Enum 값 중 하나여야 함")
    rating: int = Field(description="1점부터 5점 사이의 별점 추정치 (정수)")
    keywords: list[str] = Field(description="핵심 키워드 3개 추출")

# 3. 파서 생성
parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)
format_instructions = parser.get_format_instructions()

print("[Pydantic 파서가 자동 생성한 깐깐한 지시문]")
print(format_instructions)

[Pydantic 파서가 자동 생성한 깐깐한 지시문]
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Sentiment": {"enum": ["긍정", "부정", "중립"], "title": "Sentiment", "type": "string"}}, "properties": {"summary": {"description": "리뷰의 핵심 내용을 1문장으로 요약", "title": "Summary", "type": "string"}, "sentiment": {"$ref": "#/$defs/Sentiment", "description": "리뷰의 감정 상태. 반드시 지정된 Enum 값 중 하나여야 함"}, "rating": {"description": "1점부터 5점 사이의 별점 추정치 (정수)", "title": "Rating", "type": "integer"}, "keywords": {"description": "핵심 키워드 3개 추출", "items": {"type": "string"}, "title": "Keywords", "type": "array"}}, "required": ["summ

이제 체인을 조립하고 실제 리뷰를 던져봅니다.

In [4]:
prompt = PromptTemplate(
    template="다음 고객 리뷰를 분석해주세요.\n\n[리뷰]\n{review}\n\n{format_instructions}",
    input_variables=["review"],
    partial_variables={"format_instructions": format_instructions}
)

chain = prompt | llm | parser

review_text = "음식은 진짜 맛있는데 배달이 2시간이나 걸려서 다 식었어요. 화가 나네요."

print("[LLM 리뷰 분석 결과]")
result = chain.invoke({"review": review_text})

print("데이터 타입:", type(result))
print("결과 객체:", result)

# 완벽한 파이썬 객체이므로 자동완성(Auto-complete)과 타입힌트를 마음껏 누릴 수 있습니다.
print("\n요약:", result.summary)
print("감정:", result.sentiment.value)
print("별점:", result.rating)
print("키워드:", result.keywords)

[LLM 리뷰 분석 결과]
데이터 타입: <class '__main__.ReviewAnalysis'>
결과 객체: summary='음식은 맛있지만 배달이 매우 지연되어 식었다.' sentiment=<Sentiment.NEGATIVE: '부정'> rating=3 keywords=['맛있다', '배달', '지연']

요약: 음식은 맛있지만 배달이 매우 지연되어 식었다.
감정: 부정
별점: 3
키워드: ['맛있다', '배달', '지연']


## 6. 실패/주의 케이스: Pydantic의 무자비한 검증 (Validation Error)
모델이 Pydantic의 규칙(예: 숫자가 들어가야 하는데 문자를 넣음)을 어기면 어떻게 될까요?

In [7]:
print("[Pydantic 규칙 위반 시뮬레이션]")

# 별점에 '5'가 아니라 '오점' 이라는 한글을 강제로 집어넣은 가짜 응답
bad_json = """
{
  "summary": "맛있어요",
  "sentiment": "극도로 분노", 
  "rating": "오점",
  "keywords": ["맛"]
}
"""

try:
    # 강제로 파싱 시도 (sentiment는 Enum에 없고, rating은 int가 아님)
    parser.parse(bad_json)
except Exception as e:
    print("🔥 에러 발생:", type(e).__name__)
    print("\n에러 메시지 상세:")
    print(e)
    print("\n-> 주의: Pydantic은 '극도로 분노'가 Enum에 없다는 것, '오점'이 유효한 정수가 아니라는 것을 정확히 짚어내며 파싱을 거부합니다.")

[Pydantic 규칙 위반 시뮬레이션]
🔥 에러 발생: OutputParserException

에러 메시지 상세:
Failed to parse ReviewAnalysis from completion {"summary": "맛있어요", "sentiment": "극도로 분노", "rating": "오점", "keywords": ["맛"]}. Got: 2 validation errors for ReviewAnalysis
sentiment
  Input should be '긍정', '부정' or '중립' [type=enum, input_value='극도로 분노', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum
rating
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='오점', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

-> 주의: Pydantic은 '극도로 분노'가 Enum에 없다는 것, '오점'이 유효한 정수가 아니라는 것을 정확히 짚어내며 파싱을 거부합니다.


## 7. 결과 해석

1. **타입 안정성 보장**: `Pydantic` 클래스를 통과한 데이터는 애플리케이션(FastAPI, Django 등)에서 100% 믿고 쓸 수 있는 안전한 데이터입니다.
2. **프롬프트 엔진으로서의 Field**: `Field(description="...")`는 단순 주석이 아닙니다. 파서가 이 설명을 읽고 LLM에게 "이 칸에는 이런 내용을 적어!"라고 훌륭한 프롬프트를 알아서 생성해냅니다.
3. **에러 발생의 한계**: 여전히 LLM이 텍스트를 먼저 생성하고 그걸 나중에 Pydantic에 구겨 넣는 방식이므로, LLM이 말을 안 들으면 에러가 터지는 것은 막을 수 없습니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `JsonOutputParser`의 불안정함을 Pydantic의 강력한 타입 검증으로 보완함.
* 특히 `Enum`을 사용하니 LLM이 창의성을 발휘해서 "개빡침", "완전 좋음" 이런 식으로 대답하는 것을 원천 차단하고 "긍정/부정/중립"으로만 대답하게 묶어둘 수 있었음.
* 그러나 여전히 텍스트 파싱 기반이라 에러 확률이 존재함.

### 다음 개선 방향
* "LLM이 헛소리를 뱉은 다음 에러를 내는 것"이 아니라, 아예 "API 호출을 할 때부터 이 스키마(Schema)대로 뱉지 않으면 응답을 안 주게 강제"하는 최신 API 레벨의 기능인 `with_structured_output()`을 써볼 차례임.